In [ ]:
!pip install -q langchain langchain-openai openai chromadb gradio tiktoken langchain-community langchain-chroma

In [ ]:
!pip install --upgrade openai
from openai import OpenAI
from google.colab import userdata
api = userdata.get('openaiapi')
open_ai_client = OpenAI(api_key=api, base_url="https://openrouter.ai/api/v1")


In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


/tmp/ipykernel_1154/1753290852.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [4]:
DATA_FILE_PATH = "nobu_new_cairo_data.txt"
print(f"Data file path set to: {DATA_FILE_PATH}")

Data file path set to: nobu_new_cairo_data.txt


In [6]:
print(f"Attempting to load data from: {DATA_FILE_PATH}")
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
raw_documents = loader.load()
print(f"Loaded {len(raw_documents)} documents from: {DATA_FILE_PATH}")

Attempting to load data from: nobu_new_cairo_data.txt
Loaded 1 documents from: nobu_new_cairo_data.txt


In [7]:
print(raw_documents[0].page_content[:500])

Source: https://noburestaurants.com/new-cairo/home
Title: Nobu New Cairo
Content:
NOBU NEW CAIRO We are open for bookings. Be among the first to try Nobu’s iconic dishes in New Cairo. Secrets from the Kitchen If you are dining at Nobu for the first time, the Chef recommends trying 3 or more of the menu's "Eight Highlight Dishes." The best way to enjoy this experience is to start with 2 or 3 cold dishes, then move on to 2 or 3 hot ones. Finally, end with some sushi and dessert. All dishes are sha


In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
documents = text_splitter.split_documents(raw_documents)
if not documents:
    raise ValueError("Error: Splitting resulted in zero documents. Check the input file and splitter settings.")
print(f"Document split into {len(documents)} chunks.")

Document split into 11 chunks.


In [9]:
documents

[Document(metadata={'source': 'nobu_new_cairo_data.txt'}, page_content='Source: https://noburestaurants.com/new-cairo/home\nTitle: Nobu New Cairo\nContent:\nNOBU NEW CAIRO We are open for bookings. Be among the first to try Nobu’s iconic dishes in New Cairo. Secrets from the Kitchen If you are dining at Nobu for the first time, the Chef recommends trying 3 or more of the menu\'s "Eight Highlight Dishes." The best way to enjoy this experience is to start with 2 or 3 cold dishes, then move on to 2 or 3 hot ones. Finally, end with some sushi and dessert. All dishes are shared family style.\n---END OF SOURCE---'),
 Document(metadata={'source': 'nobu_new_cairo_data.txt'}, page_content='Source: https://noburestaurants.com/new-cairo/menus\nTitle: Nobu Menu\nContent:'),
 Document(metadata={'source': 'nobu_new_cairo_data.txt'}, page_content='Dinner Menu COLD DISHES Edamame Salted 360 Spicy Edamame 395 Baby Corn Honey Truffle 475 Umami Chicken Wings 550 Langoustine Shiso Salsa 4250 NORI TACOS Tu

In [10]:
print(documents[0].page_content)

Source: https://noburestaurants.com/new-cairo/home
Title: Nobu New Cairo
Content:
NOBU NEW CAIRO We are open for bookings. Be among the first to try Nobu’s iconic dishes in New Cairo. Secrets from the Kitchen If you are dining at Nobu for the first time, the Chef recommends trying 3 or more of the menu's "Eight Highlight Dishes." The best way to enjoy this experience is to start with 2 or 3 cold dishes, then move on to 2 or 3 hot ones. Finally, end with some sushi and dessert. All dishes are shared family style.
---END OF SOURCE---


In [12]:
print("Initializing OpenRouter Embeddings model...")

embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    api_key=api,
    base_url="https://openrouter.ai/api/v1"
)

print("Embeddings model initialized.")

print("\nCreating ChromaDB vector store and embedding documents...")

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="nobu_new_cairo_rag"
)

vector_count = vector_store._collection.count()
print(f"ChromaDB vector store created with {vector_count} items.")

if vector_count == 0:
    raise ValueError("Vector store creation resulted in 0 items.")

Initializing OpenRouter Embeddings model...
Embeddings model initialized.

Creating ChromaDB vector store and embedding documents...
ChromaDB vector store created with 11 items.


In [13]:
stored_data = vector_store._collection.get(
    include=["documents"]
)

print("Documents stored:", len(stored_data["documents"]))
print("Unique documents:", len(set(stored_data["documents"])))

Documents stored: 11
Unique documents: 10


In [14]:
stored_data = vector_store._collection.get(include=["embeddings", "documents"], limit = 1)
# Display the results
print("First chunk text:\n", stored_data['documents'][0])
print("\nEmbedding vector:\n", stored_data['embeddings'][0])
print(f"\nFull embedding has {len(stored_data['embeddings'][0])} dimensions.")

First chunk text:
 Source: https://noburestaurants.com/new-cairo/home
Title: Nobu New Cairo
Content:
NOBU NEW CAIRO We are open for bookings. Be among the first to try Nobu’s iconic dishes in New Cairo. Secrets from the Kitchen If you are dining at Nobu for the first time, the Chef recommends trying 3 or more of the menu's "Eight Highlight Dishes." The best way to enjoy this experience is to start with 2 or 3 cold dishes, then move on to 2 or 3 hot ones. Finally, end with some sushi and dessert. All dishes are shared family style.
---END OF SOURCE---

Embedding vector:
 [-0.04135132 -0.03231812  0.01203156 ... -0.01392365 -0.02662659
 -0.0362854 ]

Full embedding has 1536 dimensions.


In [15]:
test_question="What type of food included in the menu?"
print(f"Searching for documents similar to: '{test_question}'")

try:
  similar_docs = vector_store.similarity_search(test_question, k=2)
  print(f"\nFound {len(similar_docs)} similar documents:")

  for i,doc in enumerate(similar_docs):
        print(f"\n--- Document {i+1} ---")
        content_snippet = doc.page_content[:700].strip() + "..."
        source = doc.metadata.get("source", "Unknown Source")  # Get source from metadata
        print(f"Content Snippet: {content_snippet}")
        print(f"Source: {source}")
except Exception as e:
  print(f"Error: {e}")
  raise

Searching for documents similar to: 'What type of food included in the menu?'

Found 2 similar documents:

--- Document 1 ---
Content Snippet: Dinner Menu COLD DISHES Edamame Salted 360 Spicy Edamame 395 Baby Corn Honey Truffle 475 Umami Chicken Wings 550 Langoustine Shiso Salsa 4250 NORI TACOS Tuna Dry Miso 625 Salmon Spicy Miso 525 Lobster Wasabi Sour Cream 1100 Caviar & Avocado 1850 Wagyu Beef Spicy Ponzu 1650 COLD DISHES Crispy Rice with choice of Spicy Tuna, Salmon 750 Crispy Rice with Spicy King Crab 1200 Toro Tartare with Caviar 2450 Salmon Tartare with Caviar 1950 Yellowtail Jalapeño 1250 Tiradito 950 New Style Sashimi 950 White Fish Sashimi Dry Miso 950 Australian Wagyu A9 New Style Sashimi 1650 Salmon Tataki Karashi Su Miso 995 Australian Wagyu A9 Tataki 1650 Seafood Ceviche 1500 Sashimi Salad with Matsuhisa Dressing 1600...
Source: nobu_new_cairo_data.txt

--- Document 2 ---
Content Snippet: Dressing 850 Vegetable Hand Roll with Sesame Sauce 550 HOT DISHES Black Cod Miso 295

In [16]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("Retriever configured successfully from vector store.")
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=api,
    base_url="https://openrouter.ai/api/v1"
)
print("ChatOpenAI LLM successfully initialized.")
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant answering questions about a restaurant. "
     "Use ONLY the context below to answer. If the answer is not in the context, say you don't know.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    """Join the retrieved chunks into one block of text (this is what 'chain_type=stuff' used to do)."""
    return "\n\n".join(doc.page_content for doc in docs)

def get_sources(docs):
    """Unique sources from the retrieved chunks' metadata."""
    return ", ".join(sorted({doc.metadata.get("source", "Unknown Source") for doc in docs}))


qa_chain = (
    RunnablePassthrough.assign(source_documents = itemgetter("question") | retriever)        # retrieve chunks
    | RunnablePassthrough.assign(context = lambda x: format_docs(x["source_documents"]))     # stuff them into one string
    | RunnablePassthrough.assign(answer = prompt | llm | StrOutputParser())                  # generate the answer
    | RunnablePassthrough.assign(sources = lambda x: get_sources(x["source_documents"]))     # collect sources
)

print("RAG chain created")

Retriever configured successfully from vector store.
ChatOpenAI LLM successfully initialized.
RAG chain created


In [17]:
print("\n--- Testing the Full RAG Chain ---")
chain_test_query = "What kind of food does the restuarant give?"
print(f"Query: {chain_test_query}")
try:
    result = qa_chain.invoke({"question": chain_test_query})
    print("\n--- Answer ---")
    print(result.get("answer", "No answer generated."))

    print("\n--- Sources ---")
    print(result.get("sources", "No sources identified."))

    if "source_documents" in result:
        print("\n--- Source Document Snippets ---")
        for i, doc in enumerate(result["source_documents"]):
            content_snippet = doc.page_content[:250].strip()
            print(f"Doc {i+1}: {content_snippet}")

except Exception as e:
    print(f"\nAn error occurred while running the chain: {e}")


--- Testing the Full RAG Chain ---
Query: What kind of food does the restuarant give?

--- Answer ---
The restaurant offers a variety of dishes including cold dishes, hot dishes, plant-based options, soups, tempura, sushi, and sashimi. The menu features items such as black cod, seafood, wagyu beef, various salads, and a selection of sushi and sashimi.

--- Sources ---
nobu_new_cairo_data.txt

--- Source Document Snippets ---
Doc 1: Dressing 850 Vegetable Hand Roll with Sesame Sauce 550 HOT DISHES Black Cod Miso 2950 Black Cod Butter Lettuce 1400 Umami Chilean Seabass 2850 Chilean Sea Bass Jalapeño Dressing 2850 Umami Mediterranean Sea Bass 3250 Seafood Toban Yaki 1950 Rock Shri
Doc 2: Dinner Menu COLD DISHES Edamame Salted 360 Spicy Edamame 395 Baby Corn Honey Truffle 475 Umami Chicken Wings 550 Langoustine Shiso Salsa 4250 NORI TACOS Tuna Dry Miso 625 Salmon Spicy Miso 525 Lobster Wasabi Sour Cream 1100 Caviar & Avocado 1850 Wagy
Doc 3: Shrimp 750 Asparagus 325 Avocao 375 Broccoli 22

In [24]:
import gradio as gr
def ask_Nobu_New_Cairo_Restaurant(user_query):
    """
    Processes the user query using the RAG chain and returns formatted results.
    """
    print(f"\nProcessing Gradio query: '{user_query}'")
    if not user_query or user_query.strip() == "":
        print("--> Empty query received.")
        return "Please enter a question.", ""

    try:

        result = qa_chain.invoke({"question": user_query})
        answer = result.get("answer", "Sorry, I couldn't find an answer in the provided documents.")
        sources = result.get("sources", "No specific sources identified.")

        if sources == DATA_FILE_PATH:
            sources = f"Retrieved from: {DATA_FILE_PATH}"
        elif isinstance(sources, list):
            sources = ", ".join(list(set(sources)))

        print(f"--> Answer generated: {answer[:100].strip()}...")
        print(f"--> Sources identified: {sources}")

        return answer.strip(), sources

    except Exception as e:
        error_message = f"An error occurred: {e}"
        print(f"--> Error during chain execution: {error_message}")
        return error_message, "Error occurred"

print("\nSetting up Gradio interface...")

with gr.Blocks(theme=gr.themes.Soft(), title="Nobu New Cairo Restaurant Q&A Assistant") as demo:
    gr.Markdown(
        """
        # Nobu New Cairo Restaurant - AI Q&A Assistant 💬
        Ask questions about the restaurant based on its website data.
        The AI provides answers and cites the source document.
        """
    )

    question_input = gr.Textbox(
        label = "Your Question:",
        placeholder = "e.g., What are the opening hours on Saturday?",
        lines = 2,
    )

    with gr.Row():

        answer_output = gr.Textbox(label="Answer:", interactive=False, lines=6)
        sources_output = gr.Textbox(label="Sources:", interactive=False, lines=2)

    with gr.Row():
        submit_button = gr.Button("Ask Question", variant="primary")
        clear_button = gr.ClearButton(components=[question_input, answer_output, sources_output], value="Clear All")

    gr.Examples(
        examples=[
            "What are the different menu options and prices?",
            "where is the restaurant located?",
            "What are the opening hours of Nobu New Cairo?",
            "What does Nobu recommend for someone dining there for the first time?"
              ],
        inputs=question_input,
        cache_examples=False,
    )
    submit_button.click(fn = ask_Nobu_New_Cairo_Restaurant, inputs = question_input, outputs = [answer_output, sources_output])

print("Gradio interface defined.")
print("\nLaunching Gradio app... (Stop the kernel or press Ctrl+C in terminal to quit)")
demo.launch()


Setting up Gradio interface...


/tmp/ipykernel_1154/350588588.py:34: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Nobu New Cairo Restaurant Q&A Assistant") as demo:


Gradio interface defined.

Launching Gradio app... (Stop the kernel or press Ctrl+C in terminal to quit)
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3fb8c32518bed73379.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
